In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, classification_report
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

All imports successful


In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)


Train shape: (577347, 12)
Test shape: (247435, 11)


In [3]:
for df in [train, test]:
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']

print("Color indices added")
print("Train shape now:", train.shape)

Color indices added
Train shape now: (577347, 16)


In [4]:
# Encode spectral_type and galaxy_population
le_spectral = LabelEncoder()
le_galaxy = LabelEncoder()
le_class = LabelEncoder()

train['spectral_type_enc'] = le_spectral.fit_transform(train['spectral_type'])
train['galaxy_population_enc'] = le_galaxy.fit_transform(train['galaxy_population'])

test['spectral_type_enc'] = le_spectral.transform(test['spectral_type'])
test['galaxy_population_enc'] = le_galaxy.transform(test['galaxy_population'])

# Encode target
train['class_enc'] = le_class.fit_transform(train['class'])

print("Encoding done")
print("Class mapping:", dict(zip(le_class.classes_, le_class.transform(le_class.classes_))))
print("Spectral type mapping:", dict(zip(le_spectral.classes_, le_spectral.transform(le_spectral.classes_))))
print("Galaxy population mapping:", dict(zip(le_galaxy.classes_, le_galaxy.transform(le_galaxy.classes_))))

Encoding done
Class mapping: {'GALAXY': np.int64(0), 'QSO': np.int64(1), 'STAR': np.int64(2)}
Spectral type mapping: {'A/F': np.int64(0), 'G/K': np.int64(1), 'M': np.int64(2), 'O/B': np.int64(3)}
Galaxy population mapping: {'Blue_Cloud': np.int64(0), 'Red_Sequence': np.int64(1)}


In [5]:
# Features to use - dropping id, raw categoricals, target
# Also dropping alpha and delta based on EDA findings
features = ['u', 'g', 'r', 'i', 'z', 'redshift',
            'u_g', 'g_r', 'r_i', 'i_z',
            'spectral_type_enc', 'galaxy_population_enc']

X = train[features]
y = train['class_enc']

X_test = test[features]

# Split for validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # important for imbalanced classes
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("Features used:", features)

X_train shape: (461877, 12)
X_val shape: (115470, 12)
Features used: ['u', 'g', 'r', 'i', 'z', 'redshift', 'u_g', 'g_r', 'r_i', 'i_z', 'spectral_type_enc', 'galaxy_population_enc']


In [6]:
# Start simple - understand the problem first
dt = DecisionTreeClassifier(
    random_state=42,
    max_depth=10,        # limit depth to avoid overfitting
    min_samples_leaf=50  # each leaf needs at least 50 samples
)

dt.fit(X_train, y_train)

# Evaluate
dt_train_pred = dt.predict(X_train)
dt_val_pred = dt.predict(X_val)

dt_train_score = balanced_accuracy_score(y_train, dt_train_pred)
dt_val_score = balanced_accuracy_score(y_val, dt_val_pred)

print("Decision Tree Results:")
print(f"Train balanced accuracy: {dt_train_score:.4f}")
print(f"Val balanced accuracy:   {dt_val_score:.4f}")
print(f"Gap (overfit indicator): {dt_train_score - dt_val_score:.4f}")

print("\nClassification Report (Validation):")
print(classification_report(y_val, dt_val_pred, 
      target_names=le_class.classes_))

Decision Tree Results:
Train balanced accuracy: 0.9267
Val balanced accuracy:   0.9211
Gap (overfit indicator): 0.0056

Classification Report (Validation):
              precision    recall  f1-score   support

      GALAXY       0.96      0.97      0.96     75496
         QSO       0.95      0.95      0.95     23429
        STAR       0.88      0.85      0.86     16545

    accuracy                           0.95    115470
   macro avg       0.93      0.92      0.93    115470
weighted avg       0.95      0.95      0.95    115470



In [7]:
# See what the tree learned
dt_importance = pd.DataFrame({
    'feature': features,
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=False)

print("Decision Tree Feature Importance:")
print(dt_importance.to_string(index=False))

Decision Tree Feature Importance:
              feature  importance
                  g_r    0.400296
             redshift    0.395575
                    z    0.054437
                  r_i    0.052909
                    u    0.034546
                  u_g    0.024851
                    g    0.023258
                  i_z    0.006912
                    i    0.003682
galaxy_population_enc    0.002373
                    r    0.001160
    spectral_type_enc    0.000002


In [8]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    n_jobs=-1
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100  # print every 100 rounds
)

# Evaluate
xgb_train_pred = xgb_model.predict(X_train)
xgb_val_pred = xgb_model.predict(X_val)

xgb_train_score = balanced_accuracy_score(y_train, xgb_train_pred)
xgb_val_score = balanced_accuracy_score(y_val, xgb_val_pred)

print("\nXGBoost Results:")
print(f"Train balanced accuracy: {xgb_train_score:.4f}")
print(f"Val balanced accuracy:   {xgb_val_score:.4f}")
print(f"Gap (overfit indicator): {xgb_train_score - xgb_val_score:.4f}")

print("\nClassification Report (Validation):")
print(classification_report(y_val, xgb_val_pred,
      target_names=le_class.classes_))

[0]	validation_0-mlogloss:0.76835
[100]	validation_0-mlogloss:0.12498
[200]	validation_0-mlogloss:0.12327
[299]	validation_0-mlogloss:0.12338

XGBoost Results:
Train balanced accuracy: 0.9440
Val balanced accuracy:   0.9347
Gap (overfit indicator): 0.0093

Classification Report (Validation):
              precision    recall  f1-score   support

      GALAXY       0.97      0.97      0.97     75496
         QSO       0.96      0.96      0.96     23429
        STAR       0.89      0.88      0.88     16545

    accuracy                           0.95    115470
   macro avg       0.94      0.93      0.94    115470
weighted avg       0.95      0.95      0.95    115470



In [9]:
xgb_importance = pd.DataFrame({
    'feature': features,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("XGBoost Feature Importance:")
print(xgb_importance.to_string(index=False))

XGBoost Feature Importance:
              feature  importance
galaxy_population_enc    0.279404
             redshift    0.249797
                  g_r    0.159817
                    z    0.054223
    spectral_type_enc    0.053850
                    g    0.046498
                  r_i    0.038385
                    u    0.035895
                  u_g    0.033396
                    i    0.028367
                  i_z    0.010187
                    r    0.010181


In [10]:
print("=" * 40)
print("MODEL COMPARISON SUMMARY")
print("=" * 40)
print(f"Rule baseline:    0.8137")
print(f"Decision Tree:    {dt_val_score:.4f}")
print(f"XGBoost:          {xgb_val_score:.4f}")
print("=" * 40)

improvement_dt = dt_val_score - 0.8137
improvement_xgb = xgb_val_score - 0.8137
print(f"DT improvement over rules:  +{improvement_dt:.4f}")
print(f"XGB improvement over rules: +{improvement_xgb:.4f}")

MODEL COMPARISON SUMMARY
Rule baseline:    0.8137
Decision Tree:    0.9211
XGBoost:          0.9347
DT improvement over rules:  +0.1074
XGB improvement over rules: +0.1210


In [11]:

# Generate predictions on test set
test_pred = xgb_model.predict(X_test)

# Convert encoded predictions back to class labels
test_pred_labels = le_class.inverse_transform(test_pred)

# Create submission file
submission = pd.DataFrame({
    'id': test['id'],
    'class': test_pred_labels
})

print("Submission shape:", submission.shape)
print("\nPrediction distribution:")
print(submission['class'].value_counts())
print("\nSample submission:")
print(submission.head(10))

# Save
submission.to_csv('../data/submission_xgb_v1.csv', index=False)
print("\nSubmission saved!")


Submission shape: (247435, 2)

Prediction distribution:
class
GALAXY    162303
QSO        50322
STAR       34810
Name: count, dtype: int64

Sample submission:
       id   class
0  577347  GALAXY
1  577348  GALAXY
2  577349  GALAXY
3  577350    STAR
4  577351  GALAXY
5  577352  GALAXY
6  577353  GALAXY
7  577354    STAR
8  577355  GALAXY
9  577356  GALAXY

Submission saved!


In [12]:
# Make sure format matches sample submission
sample = pd.read_csv('../data/sample_submission.csv')

print("Sample submission format:")
print(sample.head())
print("\nOur submission format:")
print(submission.head())

print("\nShapes match:", submission.shape == sample.shape)
print("Columns match:", list(submission.columns) == list(sample.columns))

Sample submission format:
       id   class
0  577347  GALAXY
1  577348  GALAXY
2  577349  GALAXY
3  577350  GALAXY
4  577351  GALAXY

Our submission format:
       id   class
0  577347  GALAXY
1  577348  GALAXY
2  577349  GALAXY
3  577350    STAR
4  577351  GALAXY

Shapes match: True
Columns match: True
